# APS360 — Primary Model: ResumeMatchNet

Multi-branch deep neural network for resume-job matching (regression).

## Architecture

```
Branch 1 (Categorical + Numerical)
  ├─ 9 normalised numerical scalars
  ├─ one-hot: degree_level (6), result_type (5), job_position (51)
  └─ FC(512) → BN → ReLU → FC(256) → BN → ReLU

Branch 2 (Text Embeddings)
  ├─ SBERT-384: career_objective, responsibilities, edu_requirements
  ├─ Word2Vec-100: skills, skills_required, related_skills
  └─ FC(512) → BN → ReLU → FC(256) → BN → ReLU

Merge: concat → FC(512) → BN → ReLU → Dropout → FC(256) → BN → ReLU → Dropout → FC(1) → Sigmoid
```

**Loss**: MSELoss | **Optimiser**: Adam + CosineAnnealingLR | **Early stopping**: patience=10

In [1]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from dataset import ResumeDataset, split_dataset
from model import ResumeMatchNet
from train import train, _evaluate
from torch.utils.data import DataLoader

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
%matplotlib inline

PyTorch version: 2.9.1
CUDA available: False


## 1. Build / load the dataset

**First run** (`fit=True`): trains Word2Vec on the corpus and computes SBERT embeddings — takes ~5-15 min.  
**Subsequent runs** (`fit=False`): loads cached `.npy` files — takes seconds.

In [2]:
dataset = ResumeDataset(
    csv_path='../data/cleaned_resume_data.csv',
    cache_dir='../data/cache',
    fit=False,   # Set to True on first run to build embeddings
)
print(f'Branch 1 dim: {dataset.branch1_dim}')
print(f'Branch 2 dim: {dataset.branch2_dim}')
print(f'Total samples: {len(dataset)}')

Loading ../data/cleaned_resume_data.csv ...
  Loaded categorical vocab from ../data/cache/cat_vocab.pkl
  Loaded Branch 1 from ../data/cache/branch1.npy  shape=(9544, 55)
  Loaded SBERT embeddings from ../data/cache/sbert_embeddings.npy  shape=(9544, 1152)


ModuleNotFoundError: No module named 'gensim'

## 2. Inspect a sample

In [ ]:
b1, b2, y = dataset[0]
print(f'Branch 1 sample (first 10 values): {b1[:10].numpy().round(3)}')
print(f'Branch 2 sample (first 10 values): {b2[:10].numpy().round(3)}')
print(f'Target: {y.item():.4f}')

## 3. Model architecture

In [ ]:
model = ResumeMatchNet(
    branch1_dim=dataset.branch1_dim,
    branch2_dim=dataset.branch2_dim,
    hidden_dim=512,
    dropout=0.3,
)
print(model)

## 4. Train

You can adjust hyperparameters here. The training function handles checkpointing and early stopping automatically.

In [ ]:
results = train(
    csv_path='../data/cleaned_resume_data.csv',
    cache_dir='../data/cache',
    output_dir='../data/primary_model',
    epochs=50,
    batch_size=64,
    lr=1e-3,
    hidden_dim=512,
    dropout=0.3,
    patience=10,
    refit_embeddings=False,
)

## 5. Results summary

In [ ]:
print('=== Test Results ===')
print(f"RMSE: {results['test_rmse']:.4f}")
print(f"MAE:  {results['test_mae']:.4f}")
print(f"R²:   {results['test_r2']:.4f}")

## 6. Compare vs Baseline

In [ ]:
baseline_test_rmse = 0.1058
baseline_test_r2   = 0.5956

comparison = pd.DataFrame({
    'Model':   ['Random Forest (Baseline)', 'ResumeMatchNet (Primary)'],
    'Test RMSE': [baseline_test_rmse, results['test_rmse']],
    'Test R²':   [baseline_test_r2,   results['test_r2']],
})
print(comparison.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(2)
bars = ax.bar(x - 0.2, comparison['Test RMSE'], 0.35, label='Test RMSE', color='tomato')
bars2 = ax.bar(x + 0.2, comparison['Test R²'],  0.35, label='Test R²',   color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels(comparison['Model'])
ax.set_ylabel('Score')
ax.set_title('Baseline vs Primary Model')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../data/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Qualitative analysis — inspect individual predictions

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load('../data/primary_model/best_model.pt', map_location=device)
model = ResumeMatchNet(
    branch1_dim=dataset.branch1_dim,
    branch2_dim=dataset.branch2_dim,
    hidden_dim=512,
    dropout=0.3,
).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

train_set, val_set, test_set = split_dataset(dataset)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)

import torch.nn as nn
criterion = nn.MSELoss()
rmse, mae, r2, preds, targets = _evaluate(model, test_loader, device, criterion)
errors = np.abs(preds - targets)

# Load cleaned CSV to show qualitative context
df = pd.read_csv('../data/cleaned_resume_data.csv')
np.random.seed(42)
idx = np.random.default_rng(42).permutation(len(dataset))
test_idx = idx[int(len(dataset)*0.80):]

# Show 5 best and 5 worst predictions
order = np.argsort(errors)
print('=== 5 Best predictions ===')
for i in order[:5]:
    orig_idx = test_idx[i]
    row = df.iloc[orig_idx]
    print(f"  job={row['job_position_name'][:30]:30s}  "
          f"true={targets[i]:.3f}  pred={preds[i]:.3f}  err={errors[i]:.3f}")

print('\n=== 5 Worst predictions ===')
for i in order[-5:][::-1]:
    orig_idx = test_idx[i]
    row = df.iloc[orig_idx]
    print(f"  job={row['job_position_name'][:30]:30s}  "
          f"true={targets[i]:.3f}  pred={preds[i]:.3f}  err={errors[i]:.3f}")

## 8. Ablation: Branch contributions
Zeroing out each branch independently to measure its contribution.

In [3]:
model.eval()
ablation_results = {}

with torch.no_grad():
    all_preds_full, all_preds_b1only, all_preds_b2only = [], [], []
    all_targets = []
    for b1, b2, y in test_loader:
        b1, b2 = b1.to(device), b2.to(device)
        zeros1 = torch.zeros_like(b1)
        zeros2 = torch.zeros_like(b2)
        all_preds_full.append(model(b1, b2).cpu().numpy())
        all_preds_b1only.append(model(b1, zeros2).cpu().numpy())
        all_preds_b2only.append(model(zeros1, b2).cpu().numpy())
        all_targets.append(y.numpy())

all_preds_full   = np.concatenate(all_preds_full)
all_preds_b1only = np.concatenate(all_preds_b1only)
all_preds_b2only = np.concatenate(all_preds_b2only)
all_targets      = np.concatenate(all_targets)

def _r2(t, p):
    return 1 - np.sum((t-p)**2) / (np.sum((t-t.mean())**2) + 1e-12)

def _rmse(t, p):
    return np.sqrt(np.mean((t-p)**2))

abl_df = pd.DataFrame({
    'Configuration':  ['Full model', 'Branch 1 only (cat+num)', 'Branch 2 only (text)'],
    'RMSE': [_rmse(all_targets, all_preds_full),
             _rmse(all_targets, all_preds_b1only),
             _rmse(all_targets, all_preds_b2only)],
    'R²':   [_r2(all_targets, all_preds_full),
             _r2(all_targets, all_preds_b1only),
             _r2(all_targets, all_preds_b2only)],
})
print(abl_df.round(4).to_string(index=False))

NameError: name 'model' is not defined